# SuperWASP Pulsator Detection — Model Training

**Author:** Ana Müller, Munich  
**Dataset:** Zenodo VeSPA — McMaster et al. 2021, Res. Notes AAS 5 228  
**DOI:** [10.5281/zenodo.14937227](https://doi.org/10.5281/zenodo.14937227)

**Results:**
| Phase | F1-Score | Accuracy |
|-------|----------|----------|
| Phase 1 (classifier only) | 0.694 | 69.9% |
| Phase 2 (fine-tuning) | **0.802** | **80.2%** |

> **Enable GPU:** Runtime → Change runtime type → GPU (T4)

---
## STEP 1: Install libraries

In [ ]:
!pip install scikit-learn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import transforms, models
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import os
import copy
import random
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from google.colab import drive

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Libraries loaded!')
print(f'   Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️ No GPU — Runtime → Change runtime type → activate T4 GPU!')

---
## STEP 2: Connect Google Drive and set paths

In [ ]:
drive.mount('/content/drive')

# Paths
PULS_PFAD     = '/content/drive/MyDrive/SuperWASP/pulsatoren'
NON_PULS_PFAD = '/content/drive/MyDrive/SuperWASP/non_pulsatoren'
MODELL_PFAD   = '/content/drive/MyDrive/SuperWASP/pulsatoren_modell.pth'
ERGEBNIS_PFAD = '/content/drive/MyDrive/SuperWASP/'

# Count images
erlaubte = {'.png', '.jpg', '.jpeg'}
p_count  = len([f for f in os.listdir(PULS_PFAD) if os.path.splitext(f)[1].lower() in erlaubte])
n_count  = len([f for f in os.listdir(NON_PULS_PFAD) if os.path.splitext(f)[1].lower() in erlaubte])

print('✅ Google Drive connected!')
print(f'   Pulsators:     {p_count}')
print(f'   Non-pulsators: {n_count}')
print(f'   Total:         {p_count + n_count}')

---
## STEP 3: Load dataset with data augmentation

In [ ]:
class LichtkurvenDataset(Dataset):
    def __init__(self, transform=None):
        self.transform = transform
        self.bilder    = []
        self.labels    = []
        self.klassen   = ['non_pulsatoren', 'pulsatoren']
        erlaubte       = {'.png', '.jpg', '.jpeg'}

        for f in os.listdir(PULS_PFAD):
            if os.path.splitext(f)[1].lower() in erlaubte:
                self.bilder.append(os.path.join(PULS_PFAD, f))
                self.labels.append(1)

        for f in os.listdir(NON_PULS_PFAD):
            if os.path.splitext(f)[1].lower() in erlaubte:
                self.bilder.append(os.path.join(NON_PULS_PFAD, f))
                self.labels.append(0)

    def __len__(self):
        return len(self.bilder)

    def __getitem__(self, idx):
        try:
            bild = Image.open(self.bilder[idx]).convert('RGB')
        except Exception:
            bild = Image.new('RGB', (224, 224), color=(128, 128, 128))
        if self.transform:
            bild = self.transform(bild)
        return bild, self.labels[idx]

BILDGROESSE = 224

# Augmentation for training
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(BILDGROESSE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation without augmentation
transform_val = transforms.Compose([
    transforms.Resize((BILDGROESSE, BILDGROESSE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset
datensatz = LichtkurvenDataset(transform=transform_train)
gesamt    = len(datensatz)
train_n   = int(0.8 * gesamt)
val_n     = gesamt - train_n

# Reproducible split
torch.manual_seed(42)
train_daten, val_daten = random_split(datensatz, [train_n, val_n])

# Validation without augmentation
val_dataset = LichtkurvenDataset(transform=transform_val)
_, val_daten = random_split(val_dataset, [train_n, val_n],
    generator=torch.Generator().manual_seed(42))

# DataLoader
train_loader = DataLoader(train_daten, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_daten,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Dataset loaded!')
print(f'   Total:      {gesamt}')
print(f'   Training:   {len(train_daten)}')
print(f'   Validation: {len(val_daten)}')

---
## STEP 4: Show example images

In [ ]:
def beispiele_anzeigen(ordner, titel, anzahl=4):
    erlaubte    = {'.png', '.jpg', '.jpeg'}
    bilder_list = [f for f in os.listdir(ordner) if os.path.splitext(f)[1].lower() in erlaubte]
    auswahl     = random.sample(bilder_list, min(anzahl, len(bilder_list)))
    fig, axes   = plt.subplots(1, len(auswahl), figsize=(14, 4))
    if len(auswahl) == 1:
        axes = [axes]
    fig.suptitle(titel, fontsize=12, fontweight='bold')
    for ax, name in zip(axes, auswahl):
        bild = Image.open(os.path.join(ordner, name)).convert('RGB')
        ax.imshow(bild)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('🌟 Pulsator examples:')
beispiele_anzeigen(PULS_PFAD, 'Pulsators')
print('⭕ Non-pulsator examples:')
beispiele_anzeigen(NON_PULS_PFAD, 'Non-Pulsators')

---
## STEP 5: Build ResNet18 model with transfer learning

In [ ]:
modell = models.resnet18(pretrained=True)

# Freeze all layers
for param in modell.parameters():
    param.requires_grad = False

# Replace final layer — 2 output classes
in_features = modell.fc.in_features
modell.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 2)
)
modell = modell.to(device)

# Class-weighted loss
gewicht   = torch.tensor([p_count / n_count, 1.0]).to(device)
kriterium = nn.CrossEntropyLoss(weight=gewicht)

# Optimizer and scheduler
optimierer = optim.AdamW(modell.fc.parameters(), lr=0.001, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimierer, T_max=25)

trainierbar = sum(p.numel() for p in modell.parameters() if p.requires_grad)
print(f'✅ Model ready!')
print(f'   Architecture: ResNet18 + custom classifier')
print(f'   Trainable parameters: {trainierbar:,}')

---
## STEP 6: Phase 1 — Train classifier only

In [ ]:
EPOCHEN = 25

def trainieren(modell, train_loader, val_loader, kriterium, optimierer, scheduler, epochen):
    bestes_modell     = copy.deepcopy(modell.state_dict())
    bester_f1         = 0.0
    beste_genauigkeit = 0.0
    geduld            = 0
    MAX_GEDULD        = 8
    verlauf = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    print('='*55)
    print(f'TRAINING — {epochen} epochs')
    print('='*55)

    for epoche in range(epochen):
        # Training phase
        modell.train()
        t_loss = t_richtig = t_gesamt = 0
        for bilder, labels in train_loader:
            bilder, labels = bilder.to(device), labels.to(device)
            optimierer.zero_grad()
            out  = modell(bilder)
            loss = kriterium(out, labels)
            loss.backward()
            optimierer.step()
            t_loss    += loss.item() * bilder.size(0)
            _, pred    = torch.max(out, 1)
            t_gesamt  += labels.size(0)
            t_richtig += (pred == labels).sum().item()

        # Validation phase
        modell.eval()
        v_loss = v_richtig = v_gesamt = 0
        alle_pred  = []
        alle_label = []
        with torch.no_grad():
            for bilder, labels in val_loader:
                bilder, labels = bilder.to(device), labels.to(device)
                out   = modell(bilder)
                loss  = kriterium(out, labels)
                v_loss    += loss.item() * bilder.size(0)
                _, pred    = torch.max(out, 1)
                v_gesamt  += labels.size(0)
                v_richtig += (pred == labels).sum().item()
                alle_pred.extend(pred.cpu().numpy())
                alle_label.extend(labels.cpu().numpy())

        t_acc = t_richtig / t_gesamt
        v_acc = v_richtig / v_gesamt
        v_f1  = f1_score(alle_label, alle_pred, average='weighted', zero_division=0)
        scheduler.step()

        verlauf['train_loss'].append(t_loss / len(train_loader.dataset))
        verlauf['val_loss'].append(v_loss   / len(val_loader.dataset))
        verlauf['train_acc'].append(t_acc)
        verlauf['val_acc'].append(v_acc)
        verlauf['val_f1'].append(v_f1)

        # Save best model based on F1-score
        if v_f1 > bester_f1:
            bester_f1         = v_f1
            beste_genauigkeit = v_acc
            bestes_modell     = copy.deepcopy(modell.state_dict())
            geduld            = 0
            marker = '⭐'
        else:
            geduld += 1
            marker = ''

        print(f'E{epoche+1:2d}/{epochen} | Train {t_acc:.1%} | Val {v_acc:.1%} | F1 {v_f1:.3f} {marker}')

        # Early stopping
        if geduld >= MAX_GEDULD:
            print(f'\n⏹ Early stopping after epoch {epoche+1}')
            break

    print('='*55)
    print(f'✅ Best accuracy: {beste_genauigkeit:.1%}')
    print(f'   Best F1-score: {bester_f1:.3f}')
    modell.load_state_dict(bestes_modell)
    return modell, verlauf

modell, verlauf = trainieren(modell, train_loader, val_loader, kriterium, optimierer, scheduler, EPOCHEN)

---
## STEP 6b: Phase 2 — Fine-tuning (layer3 + layer4 + classifier)

Unfreezing deeper layers with a lower learning rate improved F1 from 0.694 to **0.802**.

In [ ]:
# Fine-tuning — unfreeze more layers
print('🔧 Fine-tuning starts...')

# Unfreeze layer3, layer4 and classifier
for name, param in modell.named_parameters():
    if any(x in name for x in ['layer3', 'layer4', 'fc']):
        param.requires_grad = True

trainierbar = sum(p.numel() for p in modell.parameters() if p.requires_grad)
print(f'   Trainable parameters now: {trainierbar:,}')

# Lower learning rate for fine-tuning
optimierer_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, modell.parameters()),
    lr=0.00005,
    weight_decay=1e-4
)
scheduler_ft = optim.lr_scheduler.CosineAnnealingLR(optimierer_ft, T_max=20)

# Train again
modell, verlauf_ft = trainieren(
    modell, train_loader, val_loader,
    kriterium, optimierer_ft, scheduler_ft, epochen=20
)

---
## STEP 7: Evaluation — metrics and visualizations

In [ ]:
modell.eval()
alle_pred  = []
alle_label = []
alle_prob  = []

with torch.no_grad():
    for bilder, labels in val_loader:
        bilder = bilder.to(device)
        out    = modell(bilder)
        prob   = torch.softmax(out, dim=1)
        _, pred = torch.max(out, 1)
        alle_pred.extend(pred.cpu().numpy())
        alle_label.extend(labels.numpy())
        alle_prob.extend(prob.cpu().numpy())

klassen_namen = ['Non-Pulsator', 'Pulsator']
genauigkeit   = sum(p == l for p, l in zip(alle_pred, alle_label)) / len(alle_label)
precision     = precision_score(alle_label, alle_pred, average='weighted', zero_division=0)
recall        = recall_score(alle_label, alle_pred, average='weighted', zero_division=0)
f1            = f1_score(alle_label, alle_pred, average='weighted', zero_division=0)

print('='*50)
print('RESULTS')
print('='*50)
print(f'  Accuracy:  {genauigkeit:.1%}')
print(f'  Precision: {precision:.1%}')
print(f'  Recall:    {recall:.1%}')
print(f'  F1-Score:  {f1:.3f}')
print('='*50)
print()
print(classification_report(alle_label, alle_pred, target_names=klassen_namen, zero_division=0))

# Visualizations
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)
fig.suptitle('SuperWASP Pulsator Detection — Results', fontsize=14, fontweight='bold')

# Loss curve
ax1 = fig.add_subplot(gs[0, 0])
ep  = range(1, len(verlauf['train_loss']) + 1)
ax1.plot(ep, verlauf['train_loss'], 'b-', label='Training', linewidth=2)
ax1.plot(ep, verlauf['val_loss'],   'r-', label='Validation', linewidth=2)
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ep, [a*100 for a in verlauf['train_acc']], 'b-', label='Training', linewidth=2)
ax2.plot(ep, [a*100 for a in verlauf['val_acc']],   'r-', label='Validation', linewidth=2)
ax2.set_title('Accuracy %')
ax2.set_ylim([0, 100])
ax2.legend()
ax2.grid(True, alpha=0.3)

# F1-score curve
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(ep, verlauf['val_f1'], 'g-', linewidth=2)
ax3.set_title('F1-Score')
ax3.set_ylim([0, 1])
ax3.grid(True, alpha=0.3)

# Confusion matrix
ax4 = fig.add_subplot(gs[1, 0])
cm  = confusion_matrix(alle_label, alle_pred)
im  = ax4.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax4)
ax4.set_xticks(range(2))
ax4.set_yticks(range(2))
ax4.set_xticklabels(klassen_namen, rotation=45, ha='right')
ax4.set_yticklabels(klassen_namen)
for i in range(2):
    for j in range(2):
        ax4.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black',
                fontsize=14, fontweight='bold')
ax4.set_title('Confusion Matrix')
ax4.set_ylabel('True label')
ax4.set_xlabel('Predicted')

# Metrics bar chart
ax5      = fig.add_subplot(gs[1, 1])
metriken = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
werte    = [genauigkeit, precision, recall, f1]
farben   = ['#2E75B6', '#C9A227', '#1E6B3C', '#8B0000']
bars     = ax5.bar(metriken, werte, color=farben)
ax5.set_ylim([0, 1])
ax5.set_title('Metrics')
for bar, wert in zip(bars, werte):
    ax5.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{wert:.1%}', ha='center', fontweight='bold')
ax5.set_xticklabels(metriken, rotation=20, ha='right')
ax5.grid(True, alpha=0.3, axis='y')

# Class distribution
ax6 = fig.add_subplot(gs[1, 2])
ax6.pie([n_count, p_count],
        labels=['Non-Pulsator', 'Pulsator'],
        colors=['#2E75B6', '#C9A227'],
        autopct='%1.1f%%', startangle=90)
ax6.set_title(f'Dataset ({p_count + n_count} images)')

plt.tight_layout()
plt.savefig(os.path.join(ERGEBNIS_PFAD, 'ergebnisse.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure saved!')

---
## STEP 8: Save model

In [ ]:
torch.save({
    'modell_zustand':    modell.state_dict(),
    'klassen':           ['non_pulsatoren', 'pulsatoren'],
    'architektur':       'resnet18',
    'bildgroesse':       BILDGROESSE,
    'beste_genauigkeit': genauigkeit,
    'bester_f1':         f1,
    'precision':         precision,
    'recall':            recall,
    'trainingsbilder':   len(train_daten),
    'validierungsbilder':len(val_daten),
}, MODELL_PFAD)

print(f'✅ Model saved!')
print(f'   File: pulsatoren_modell.pth')
print(f'   Accuracy: {genauigkeit:.1%}')
print(f'   F1-Score: {f1:.3f}')

---
## STEP 9: Classify a new light curve image

In [ ]:
# ═══════════════════════════════════════════
# ADJUST HERE: path to test image
# ═══════════════════════════════════════════
TESTBILD = '/content/drive/MyDrive/SuperWASP/testbild.png'
# ═══════════════════════════════════════════

if os.path.exists(TESTBILD):
    modell.eval()
    bild   = Image.open(TESTBILD).convert('RGB')
    tensor = transform_val(bild).unsqueeze(0).to(device)

    with torch.no_grad():
        out  = modell(tensor)
        prob = torch.softmax(out, dim=1)
        _, pred = torch.max(out, 1)

    klasse    = 'PULSATOR' if pred.item() == 1 else 'NON-PULSATOR'
    konfidenz = prob[0][pred.item()].item()
    p_prob    = prob[0][1].item()
    n_prob    = prob[0][0].item()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(bild)
    farbe = '#1E6B3C' if pred.item() == 1 else '#8B0000'
    ax1.set_title(f'{klasse}\nConfidence: {konfidenz:.1%}',
                 fontsize=14, fontweight='bold', color=farbe)
    ax1.axis('off')

    ax2.barh(['Non-Pulsator', 'Pulsator'], [n_prob, p_prob],
            color=['#2E75B6', '#C9A227'])
    ax2.set_xlim([0, 1])
    ax2.set_title('Class probability')
    for i, v in enumerate([n_prob, p_prob]):
        ax2.text(v + 0.01, i, f'{v:.1%}', va='center', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    print(f'🔭 Result: {klasse}')
    print(f'   Pulsator:     {p_prob:.1%}')
    print(f'   Non-pulsator: {n_prob:.1%}')
else:
    print(f'ℹ️  No test image found.')
    print(f'   Place an image here: {TESTBILD}')

---
## Done!

**Saved to Google Drive:**
- `pulsatoren_modell.pth` — trained model
- `ergebnisse.png` — results figure

**Results:**
| Metric | Phase 1 | Phase 2 (Final) |
|--------|---------|-----------------|
| F1-Score | 0.694 | **0.802** |
| Accuracy | 69.9% | **80.2%** |

**Dataset:** 3,999 SuperWASP VeSPA light curve images  
**Citation:** McMaster et al. 2021, Res. Notes AAS 5 228. DOI: 10.5281/zenodo.14937227